# Naive RAG Walkthrough
투자 트랜스크립트 -> pgvector -> gpt-4o-mini.

In [ ]:
from naive_rag.loader import load_transcripts
docs = load_transcripts(limit=3)
print(f"Loaded {len(docs)} documents")
docs[0].metadata, docs[0].page_content[:300]

## Split

In [ ]:
from naive_rag.splitter import split_documents
chunks = split_documents(docs)
print(f"{len(docs)} docs -> {len(chunks)} chunks")
chunks[0].page_content[:200], chunks[0].metadata

## Embed (single query for inspection)

In [ ]:
from naive_rag.embeddings import build_embeddings
emb = build_embeddings()
vec = emb.embed_query("매크로 지표")
print(f"dim={len(vec)}, first 5={vec[:5]}")

## Retrieve

In [ ]:
from naive_rag.retriever import build_retriever
retriever = build_retriever()
hits = retriever.invoke("최근 자주 인용되는 매크로 지표는?")
for i, h in enumerate(hits):
    print(f"--- hit {i} (channel={h.metadata.get('channel')}) ---")
    print(h.page_content[:200])

## Generate

In [ ]:
from naive_rag.chain import build_chain
chain = build_chain()
answer = chain.invoke("최근 트랜스크립트에서 자주 인용되는 매크로 지표는?")
print(answer)

## Run all 10 eval questions

In [ ]:
questions = [
    "최근 트랜스크립트에서 언급된 주요 매수 근거는 무엇인가?",
    "자주 인용되는 매크로 지표는 무엇인가?",
    "반증 조건(thesis가 깨지는 조건)으로 제시된 내용은?",
    "특정 종목에 대한 방향성(상승/하락)과 시간 지평은?",
    "최근 콘텐츠에서 위험 요소로 언급된 것은?",
    "특정 섹터(반도체/2차전지/AI 등)에 대한 시각은?",
    "두 인플루언서의 의견이 갈리는 지점은?",
    "동일 종목에 대해 서로 다른 근거를 제시한 사례는?",
    "미국 금리/연준 정책에 대한 견해는?",
    "환율(원/달러)에 대한 언급과 그 영향은?",
]
for i, q in enumerate(questions, 1):
    print(f"\n=== Q{i}. {q}")
    print(chain.invoke(q))

## Ragas Evaluation

4 metrics: **Faithfulness**, **AnswerRelevancy**, **LLMContextPrecisionWithoutReference**, **ContextEntityRecall**.

ContextEntityRecall은 reference(정답)에서 entity를 추출해 contexts에 얼마나 들어있는지 본다. 아래 `eval_data` 리스트에서 각 질문의 `reference` 값을 직접 입력. Reference는 핵심 entity(종목명, 인물, 지표명 등)가 들어있는 1-3문장이 좋다.

Judge LLM은 gpt-4o-mini, judge embeddings는 bge-m3.

In [ ]:
from datasets import Dataset
from langchain_openai import ChatOpenAI
from ragas import evaluate
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import (
    AnswerRelevancy,
    ContextEntityRecall,
    Faithfulness,
    LLMContextPrecisionWithoutReference,
)

from naive_rag.chain import build_chain
from naive_rag.config import Settings
from naive_rag.embeddings import build_embeddings
from naive_rag.retriever import build_retriever

# 사용자 입력: 각 질문에 대한 reference(정답)를 직접 작성.
# 핵심 entity(종목명, 인물, 지표명, 섹터 등)가 포함된 1-3문장 권장.
eval_data = [
    {
        "question": "최근 트랜스크립트에서 언급된 주요 매수 근거는 무엇인가?",
        "reference": "",
    },
    {
        "question": "자주 인용되는 매크로 지표는 무엇인가?",
        "reference": "",
    },
    {
        "question": "반증 조건(thesis가 깨지는 조건)으로 제시된 내용은?",
        "reference": "",
    },
    {
        "question": "특정 종목에 대한 방향성(상승/하락)과 시간 지평은?",
        "reference": "",
    },
    {
        "question": "최근 콘텐츠에서 위험 요소로 언급된 것은?",
        "reference": "",
    },
    {
        "question": "특정 섹터(반도체/2차전지/AI 등)에 대한 시각은?",
        "reference": "",
    },
    {
        "question": "두 인플루언서의 의견이 갈리는 지점은?",
        "reference": "",
    },
    {
        "question": "동일 종목에 대해 서로 다른 근거를 제시한 사례는?",
        "reference": "",
    },
    {
        "question": "미국 금리/연준 정책에 대한 견해는?",
        "reference": "",
    },
    {
        "question": "환율(원/달러)에 대한 언급과 그 영향은?",
        "reference": "",
    },
]

# Reference 미입력 항목 체크
missing = [d["question"] for d in eval_data if not d["reference"].strip()]
if missing:
    raise ValueError(
        f"{len(missing)}개 질문의 reference가 비어 있다. eval_data를 채운 뒤 다시 실행:\n"
        + "\n".join(f"  - {q}" for q in missing)
    )

retriever = build_retriever()
chain = build_chain()
rows = [
    {
        "user_input": d["question"],
        "reference": d["reference"],
        "response": chain.invoke(d["question"]),
        "retrieved_contexts": [c.page_content for c in retriever.invoke(d["question"])],
    }
    for d in eval_data
]

settings = Settings()
result = evaluate(
    Dataset.from_list(rows),
    metrics=[
        Faithfulness(),
        AnswerRelevancy(),
        LLMContextPrecisionWithoutReference(),
        ContextEntityRecall(),
    ],
    llm=LangchainLLMWrapper(
        ChatOpenAI(
            model=settings.llm_model,
            api_key=settings.openai_api_key,
            temperature=0,
        )
    ),
    embeddings=LangchainEmbeddingsWrapper(build_embeddings()),
)
result.to_pandas()